# 02 - Data exploration

This notebook displays example documents and a few basic statistics for each dataset. It does not modify or preprocess the data. Run `01_data_loading.ipynb` first.

## 1. Install packages

In [ ]:
%pip install -q "datasets>=3.0,<5" "matplotlib>=3.8,<4" "ipywidgets>=8,<9"

## 2. Load the downloaded datasets

In [ ]:
from collections import Counter
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
from datasets import load_from_disk

SEED = 566


def is_repo_root(path: Path) -> bool:
    return (path / "README.md").exists() and (path / "notebooks").exists()


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current / "poisoned-paperwork"]
    for parent in current.parents:
        candidates.extend([parent, parent / "poisoned-paperwork"])
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate
    raise RuntimeError("Could not find the local poisoned-paperwork repository.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"

required_paths = [
    DATA_DIR / "sroie",
    DATA_DIR / "cord_v2",
    DATA_DIR / "resume_parsing_vision",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing dataset folders. Run 01_data_loading.ipynb first:\n"
        + "\n".join(missing)
    )

sroie = load_from_disk(DATA_DIR / "sroie")
cord = load_from_disk(DATA_DIR / "cord_v2")
resumes = load_from_disk(DATA_DIR / "resume_parsing_vision")

print("SROIE:", {split: len(data) for split, data in sroie.items()})
print("CORD v2:", {split: len(data) for split, data in cord.items()})
print("English resumes:", {split: len(data) for split, data in resumes.items()})

# SROIE

SROIE contains English receipt images with OCR words, bounding boxes, and labeled fields such as company, date, address, and total.

In [ ]:
sample_indices = random.Random(SEED).sample(range(len(sroie["train"])), 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for axis, index in zip(axes, sample_indices):
    example = sroie["train"][index]
    total = example.get("entities", {}).get("total", "missing")
    axis.imshow(example["image"])
    axis.set_title(f"Index {index}\nTotal: {total}")
    axis.axis("off")

fig.suptitle("SROIE receipt examples")
plt.tight_layout()
plt.show()

In [ ]:
split_names = list(sroie.keys())
split_counts = [len(sroie[name]) for name in split_names]
word_counts = [len(words) for split in sroie.values() for words in split["words"]]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(split_names, split_counts, color="steelblue")
axes[0].set_title("Documents per split")
axes[0].set_ylabel("Documents")
axes[1].hist(word_counts, bins=25, color="steelblue", edgecolor="white")
axes[1].set_title("OCR words per receipt")
axes[1].set_xlabel("Number of words")
axes[1].set_ylabel("Receipts")
plt.tight_layout()
plt.show()

# CORD v2

CORD v2 contains photographed Indonesian receipts with structured JSON annotations for menu items, subtotals, and totals.

In [ ]:
def cord_total(example):
    annotation = example["ground_truth"]
    if isinstance(annotation, str):
        annotation = json.loads(annotation)
    total = annotation.get("gt_parse", {}).get("total", {})
    if isinstance(total, list):
        total = total[0] if total else {}
    return total.get("total_price", "missing") if isinstance(total, dict) else "missing"


sample_indices = random.Random(SEED).sample(range(len(cord["train"])), 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for axis, index in zip(axes, sample_indices):
    example = cord["train"][index]
    axis.imshow(example["image"])
    axis.set_title(f"Index {index}\nTotal: {cord_total(example)}")
    axis.axis("off")

fig.suptitle("CORD v2 receipt examples")
plt.tight_layout()
plt.show()

In [ ]:
cord_split_names = list(cord.keys())
cord_split_counts = [len(cord[name]) for name in cord_split_names]
receipt_heights = []

for split in cord.values():
    for raw_annotation in split["ground_truth"]:
        annotation = json.loads(raw_annotation) if isinstance(raw_annotation, str) else raw_annotation
        height = annotation.get("meta", {}).get("image_size", {}).get("height")
        if height is not None:
            receipt_heights.append(height)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(cord_split_names, cord_split_counts, color="darkorange")
axes[0].set_title("Documents per split")
axes[0].set_ylabel("Documents")
axes[1].hist(receipt_heights, bins=25, color="darkorange", edgecolor="white")
axes[1].set_title("Receipt image heights")
axes[1].set_xlabel("Height in pixels")
axes[1].set_ylabel("Receipts")
plt.tight_layout()
plt.show()

# English synthetic resumes

This dataset contains 1,000 synthetic English resumes. Each example contains one to three rendered page images and structured JSON ground truth, including education degree labels.

In [ ]:
def resume_ground_truth(example):
    value = example["ground_truth"]
    return json.loads(value) if isinstance(value, str) else value


def resume_degrees(example):
    education = resume_ground_truth(example).get("educations", [])
    return [item.get("degree") for item in education if item.get("degree")]


sample_indices = random.Random(SEED).sample(range(len(resumes["train"])), 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for axis, index in zip(axes, sample_indices):
    example = resumes["train"][index]
    axis.imshow(example["images"][0])
    axis.set_title(
        f"{example['sample_id']} - page 1/{example['num_pages']}\n"
        f"Degrees: {resume_degrees(example)}"
    )
    axis.axis("off")

fig.suptitle("English synthetic resume examples")
plt.tight_layout()
plt.show()

In [ ]:
resume_split_names = list(resumes.keys())
resume_split_counts = [len(resumes[name]) for name in resume_split_names]
page_counts = []
degree_counts = Counter()

for split in resumes.values():
    page_counts.extend(split["num_pages"])
    for raw_ground_truth in split["ground_truth"]:
        ground_truth = json.loads(raw_ground_truth) if isinstance(raw_ground_truth, str) else raw_ground_truth
        degree_counts.update(
            item.get("degree")
            for item in ground_truth.get("educations", [])
            if item.get("degree")
        )

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(resume_split_names, resume_split_counts, color="seagreen")
axes[0].set_title("Documents per split")
axes[0].set_ylabel("Resumes")

page_distribution = Counter(page_counts)
page_numbers = sorted(page_distribution)
axes[1].bar(page_numbers, [page_distribution[x] for x in page_numbers], color="seagreen")
axes[1].set_title("Pages per resume")
axes[1].set_xlabel("Pages")
axes[1].set_ylabel("Resumes")

top_degrees = degree_counts.most_common(10)
labels = [degree for degree, _ in reversed(top_degrees)]
values = [count for _, count in reversed(top_degrees)]
axes[2].barh(labels, values, color="seagreen")
axes[2].set_title("Most common degree labels")
axes[2].set_xlabel("Occurrences")

plt.tight_layout()
plt.show()
print(f"Single-page resumes: {page_counts.count(1)}")

## Initial observations to record

After running the notebook, note anything that could affect later modeling: receipt resolution, image quality, missing totals, language differences, multi-page resumes, and inconsistent degree labels. Normalization, filtering, split construction, or merging should be implemented later in `03_data_processing.ipynb`.